In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
model = ChatOpenAI()

In [13]:
class BlogState(TypedDict):
    topic : str
    outline : str
    report : str
    evaluation : str

In [7]:
def create_outline(state:BlogState):
    topic = state['topic']
    prompt = f"Create a detailed outline for a blog on the topic - {topic}"
    outline = model.invoke(prompt).content
    return {'outline':outline}

In [8]:
def create_report(state:BlogState):
    topic = state['topic']
    outline = state['outline']
    prompt = f"Create a detailed report on the topic - {topic} based on the outline given below:\n{outline}"
    report = model.invoke(prompt).content
    return {'report':report}

In [9]:
def evaluate_report(state:BlogState):
    topic = state['topic']
    outline = state['outline']
    report = state['report']
    prompt = f"For the topic - {topic} and a relevant outline - \n{outline}, \nevaluate the report given below on a scale of 1-5 with 5 being the best.\n{report}"
    evaluation = model.invoke(prompt).content
    return {'evaluation':evaluation}

In [11]:
graph = StateGraph(BlogState)

graph.add_node('create_outline',create_outline)
graph.add_node('create_report',create_report)
graph.add_node('evaluate_report',evaluate_report)

graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_report')
graph.add_edge('create_report','evaluate_report')
graph.add_edge('evaluate_report',END)

workflow = graph.compile()

In [12]:
initial_state = {'topic':"AI in India"}
final_state = workflow.invoke(initial_state)

final_state

{'topic': 'AI in India',
 'outline': 'I. Introduction \n    A. Explanation of what AI (Artificial Intelligence) is \n    B. Overview of the growth and importance of AI in India\n    \nII. The Current State of AI in India \n    A. Statistics on the growth of AI in India \n    B. Major companies and startups using AI in India \n    C. Government initiatives to promote AI in India \n    \nIII. Applications of AI in India \n    A. Healthcare \n    B. Agriculture \n    C. Finance \n    D. Education \n    \nIV. Challenges and Opportunities \n    A. Lack of skilled professionals in AI \n    B. Data privacy and security concerns \n    C. Potential for economic growth and job creation \n    D. Ethical considerations \n    \nV. Future of AI in India \n    A. Predictions for the growth of AI in India \n    B. Potential challenges and opportunities \n    C. Role of government and industry in shaping the future of AI in India \n    \nVI. Conclusion \n    A. Recap of the importance and impact of AI 